# **Note for Colab Users**

# **Do not write directly in this file. Your work may disappear!**

# **Always make a copy before you start.**

How to make a copy

1. Click **File** in the top-left corner.  
> *If you cannot see menus like **File** or **Runtime**, click the “v” mark in the top-right corner to show them.*

2. Choose **Save a copy in Drive**.  

3. Rename the copied file to `YOURNAMEs_FileName.ipynb`.  
> Example: If your name is Olivia → `Olivias_FileName.ipynb`  


---

* Check marks (✅) are not saved. They disappear when you refresh the page with Chrome’s reload button.<br>  
If you stop halfway, add a text cell and write something like `SO FAR DONE`.

---

* In Colab, **previous output results are reset every 30 to 90 minutes**.<br>  
So errors like `~~ is not defined` happen **all the time**.

🔁 What should you do when you see a `~~ is not defined` error?

1. First, check the spelling of the variable name.<br>  
2. If the spelling is correct but the error still appears, **click that cell to select it**.<br>  
3. Click **Runtime** → **Run before** in the top-left menu.<br>  
→ This reruns **all cells up to that point**.  
4. Run that cell again.

If the error still does not go away,<br>  
there may be a basic mistake in your TODO answer in an earlier cell.<br>  
Check whether it is correct.<br>  
Or ask ChatGPT or another coding assistant for help.


# **Preparation**

In this part, we only load the content from the previous Chapter.<br>
Just run the code. You do not need to read it carefully.<br>
Feel free to move on.<br>


In [ ]:
# Download the file
!wget https://raw.githubusercontent.com/HayatoHongo/ColabGPT/main/input.txt -O input.txt
# Load the downloaded input.txt file with utf-8.
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()

# A function to display tensors in an easy-to-read format (optional)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Install a library that displays tensors in an easy-to-read format
!pip install git+https://github.com/HayatoHongo/print_formatted_tensor.git
# Import a function that displays PyTorch tensors in an easy-to-read format
from torch_print_tensor import print_formatted_tensor



# **Chapter 14: Tokens per second(T4 GPU)**



### **Section 1: Measuring `tokens_per_second`**


Google Colab provides a free T4 GPU.

In the previous Chapter, the runtime was CPU, so this time please set it to the T4 GPU.


Let’s double-check just to be safe.


In [ ]:
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'  # Device to use (GPU or CPU)
print(device_type)

**`Check Point`**  
<label><input type="checkbox">Confirmed that the runtime is set to cuda (GPU)<br></label>  


Welcome to the EveryonesAI circuit. The CPU vs GPU race has begun.

Our GPU racer is about to blast off.

This time too, we will calculate `tokens_per_second`, which tells us how many tokens can be processed in one second.

The Trainer class is exactly the same as in the previous Chapter.

It is already integrated with tokens_per_second, so you can use it right away.

Just run it.


In [ ]:
############ NEW ############
import time
############ NEW ############

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Calculate loss for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Return to training mode again

        # Calculate and return the average loss for each dataset (train, val)
        return {split: sum(values) / len(values) for split, values in losses.items()}

    def train(self):
        # Run train_step the number of times specified in config.
        for step in range(self.config.total_training_steps):

            # Evaluate every 100 times, or only on the final step.
            if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
                ########## NEW ##########
                # Exclude step==0 because last_eval_end_time is undefined. Exclude the final step because it may be a partial measurement.
                if step == 0 or step == self.config.total_training_steps - 1:
                  tokens_per_second = None
                else:
                  current_eval_start_time = time.time()
                  evaluation_interval = current_eval_start_time - last_eval_end_time
                  tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
                  tokens_per_second = tokens_per_evaluation_interval / evaluation_interval
                ########## NEW ##########

                eval_loss = self.evaluate()
                print(f"Step {step}: Train Loss {eval_loss['train']:.4f}, Validation Loss {eval_loss['val']:.4f}")

                ########## NEW ##########
                print(f"Tokens per second {tokens_per_second}")
                # Record when this evaluation finished. The time difference to the next evaluation start becomes `evaluation_interval`.
                last_eval_end_time = time.time()
                ########## NEW ##########

            # One training step (the main process done every time)
            train_loss = self.train_step()

Define the `Dataloader` class and the model class. They are exactly the same as in Chpater12.


In [ ]:
class DataLoader:
    def __init__(self, text, config):
        self.config = config  # Config object
        chars = sorted(list(set(text)))  # Sort the unique characters
        self.ctoi = {char: index for index, char in enumerate(chars)}
        self.itoc = {index: char for index, char in enumerate(chars)}
        self.vocab_size = len(chars)

        # Encode and convert to a tensor.
        # You need `self.` to call methods or arguments outside `__init__`.
        self.data = torch.tensor(self.encode(text), dtype=torch.long)

        # Split into training and validation data.
        # Even if no arguments are specified, self.data is used.
        self.train_data, self.val_data = self.split_data()

    def encode(self, text):
        # Convert a string into a sequence of indices. Use self. to call other methods or arguments.
        return [self.ctoi[c] for c in text]

    def decode(self, indices):
        return ''.join([self.itoc[i] for i in indices])

    def split_data(self):
        split_index = int(0.9 * len(self.data))  # The point where 90% of the data is split off for training.
        return self.data[:split_index], self.data[split_index:]

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        start_indices = torch.randint(len(data) - self.config.input_sequence_length, (self.config.batch_size,)) # Generate starting indices for extraction

        input_sequences = torch.stack([
            data[start_index:start_index + self.config.input_sequence_length]
            for start_index in start_indices
        ])
        target_sequences = torch.stack([
            data[start_index + 1:start_index + self.config.input_sequence_length + 1]
            for start_index in start_indices
        ])
        return input_sequences.to(self.config.device_type), target_sequences.to(self.config.device_type)


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Define an embedding table with vocab_size x embedding_dim
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        # Get the embedding vectors that correspond to the input indices
        return self.token_embedding_table.forward(input_indices)

class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length = 8, embedding_dim = 8):
        super().__init__()
        # Position embedding layer
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        # Shape of the input tensor input_indices: [batch size, sequence length].
        sequence_length = input_indices.shape[1]

        # Create position indices based on the sequence length (example: [0, 1, 2, ..., sequence_length-1])
        position_indices = torch.arange(sequence_length, device=input_indices.device)

        # Get the embedding vectors for the position indices
        position_embeddings = self.position_embedding_layer.forward(position_indices)

        return position_embeddings

class EmbeddingModule(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Embedding layer for each token
        self.token_embedding_layer = TokenEmbedding(vocab_size = vocab_size, embedding_dim = config.embedding_dim)  # Token embedding layer
        self.position_embedding_layer = PositionEmbedding(input_sequence_length = config.input_sequence_length, embedding_dim = config.embedding_dim)  # Embed position information

    def forward(self, input_indices):
        # Get token embeddings
        token_embeddings = self.token_embedding_layer.embed(input_indices)

        # Get position embeddings
        position_embeddings = self.position_embedding_layer.forward(input_indices)

        # Add token embeddings and position embeddings
        embeddings = position_embeddings + token_embeddings
        return embeddings

class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc= nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)

        # Dropout
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape  # Batch, token length, embedding channels

        Key = self.key_fc.forward(input_tensor)     # (B, T, head_size)
        Query = self.query_fc.forward(input_tensor)   # (B, T, head_size)
        Value = self.value_fc.forward(input_tensor)   # (B, T, head_size)

        # Calculate Attention scores (QK^T) / sqrt(embedding_dim)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size**(-0.5)

        # Mask applied
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        # Softmax -> dropout -> weighted sum
        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        # Manage multiple heads with ModuleList
        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config)
            for _ in range(self.num_attention_heads)
        ])

        # Linear layer that mixes the outputs of each head
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

        # Dropout for the output
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        # Get the output from each head
        # A list of (B, T, head_size) tensors
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]

        # Concatenate all head outputs -> (B, T, embedding_dim)
        concatenated = torch.cat(head_outputs_list, dim=-1)

        # Mix outputs with a linear transformation
        projected = self.output_projection.forward(concatenated)

        # Apply dropout to the final output
        output = self.dropout.forward(projected)

        return output

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Each LayerNorm keeps its own beta and gamma.
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)

        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        # The forward method is abbreviated.
        normed_input = self.layer_norm1(input_tensor) # Apply LayerNorm to the input
        attention_output = self.multihead_attention(normed_input) # Apply multi-head attention
        residual_attention = attention_output + input_tensor # Add "before! layernorm1"
        normed_attention = self.layer_norm2(residual_attention) # Apply LayerNorm again to the residual output
        feedforward_output = self.feed_forward(normed_attention) # Apply the feed-forward network
        final_output = feedforward_output + residual_attention # Add "before" layernorm2!

        return final_output

class VocabularyLogits(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Layer normalization
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        # Projection to the vocabulary size
        self.vocab_projection = nn.Linear(config.embedding_dim, vocab_size)

    def forward(self, transformer_block_output):
        # Apply Layer normalization to the Transformer block output.
        normalized_output = self.output_norm.forward(transformer_block_output)  # (B, T, C)

        # Convert scores to the vocabulary-size dimension with a linear layer.
        vocab_logits = self.vocab_projection.forward(normalized_output)  # (B, T, V)

        return vocab_logits

class nanoGPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.config = config  # Keep this because it is also used during generation.
        self.embedding = EmbeddingModule(vocab_size, config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(vocab_size=vocab_size, config=config)
        self.criterion = nn.CrossEntropyLoss()

    # Generate text
    def generate(self, input_indices, max_new_tokens):
        # Generate only the specified number of tokens, max_new_tokens
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -self.config.input_sequence_length:] # Crop the input

            # The forward pass returns `(likelihood, loss)` -- keep only `likelihood` as `logits`.
            logits, _ = self.forward(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] # Extract the logits for the last token
            probs = F.softmax(last_logits, dim=-1) # Convert likelihood to probabilities with Softmax

            # Sample the next token
            next_token = torch.multinomial(probs, num_samples=1)

            # Add the new token and update input_indices.
            input_indices = torch.cat((input_indices, next_token), dim=1)

        # Return the final `input_indices`. Its length is the original `input_indices` + `max_new_tokens`
        return input_indices

    # Calculate likelihood and loss
    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        # During inference, there is no target, so loss is None
        # -- only the probabilities (logits) are returned.
        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)

        return logits, loss


First, let’s measure with batch size 1 and no parallel computation.

Let’s start training and measure `tokens_per_second`.

First, use the following settings.





In [ ]:
# A config class that stores the model settings
class ModelConfig:
    ########## NEW ##########
    batch_size = 1  # First, set the batch size to 1. No parallel computation.
    input_sequence_length = 512  # Use a longer sequence at once to reduce the share of time spent on data loading.
    total_training_steps = 150  # This is only for measuring tokens per second, so set the max steps to around 150.
    device_type = 'cuda'  # Fix the device to GPU
    ########## NEW ##########
    evaluation_frequency = 100  # How often to evaluate model performance
    learning_rate = 0.001  # Learning rate
    evaluation_loops = 10  # Number of loops during evaluation
    embedding_dim = 64  # Embedding layer size (number of dimensions in the feature vector)
    hidden_dim = 256
    num_attention_heads = 4  # Number of attention heads
    layer_count = 4  # Number of model layers
    dropout_rate = 0.1  # Dropout probability
    random_seed_value = 1337  # Random seed for reproducibility

In [ ]:
# Load the settings and set the seed
config = ModelConfig()
torch.manual_seed(config.random_seed_value)  # Set the random seed for reproducibility

In [ ]:
# Load the data
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text_data = f.read()
data_loader = DataLoader(text_data, config)

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # Specify the device to use
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
# Display the number of model parameters
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

**`Check Point`**  
<label><input type="checkbox">Checked `tokens_per_second`<br></label>  


Even with batch size 1, the GPU is overwhelmingly faster than the CPU.

That is because a GPU can process matrix operations (lots of multiplications + additions) in parallel all at once.


The GPU has already posted a wild speed.

Next, increase the batch size from 1 to 16 and run the same calculation.


In [ ]:
config.batch_size = # TODO: Try increasing the batch size from 1 to 16

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # Specify the device to use
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

Because the GPU can do parallel computation, the processing time is not proportional to the batch size.  

That is why it can reach incredible speed.

Click the 🔽 next to **RAM / Disk** in the top-right corner and open **View resources**.


In addition to **System RAM**, there is also **GPU RAM**. This lets you check the current GPU memory usage.

When you increase the batch size, the amount of data (tensors) held at once also increases,  
so memory usage also goes up. This is the same as with the CPU.

Let’s increase the batch size a little more.


In [ ]:
config.batch_size = 128

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # Specify the device to use
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

Wait... 😳 It did not get that much faster, did it?

Yes, exactly.

Check the GPU memory usage.

It has gone up a lot.

When memory usage starts very low, like when we increased batch size from 1 to 16, raising the usage can dramatically increase `tokens per minute`. But after memory usage reaches a certain level, it does not increase much.

We will skip the detailed reason, but that is how it behaves.

In other words, pushing the batch size all the way to the limit does not give a big benefit.


**Section 1: Measuring tokens_per_second** <label><input type="checkbox"> Mark as Done</label>


### **Section 2: Out Of Memory Error**

Now let’s try to use up all the memory.

What happens if we make the batch size extremely large?


In [ ]:
config.batch_size = 1024

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # Specify the device to use
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

When GPU memory runs out, you get an OOM (Out Of Memory) error.


The CPU can automatically split an oversized batch into smaller pieces,  
but the GPU tries to **load everything at once**, so OOM happens when there is not enough memory.

---

### ⚙️ Fixes and Rough Guidelines

| Item | Description |
|------|-------------|
| **Error name** | `OutOfMemoryError` (OOM for short) |
| **Cause** | The batch size or sequence length is too large |
| **Fix** | Reduce the batch size / shorten the input |
| **Rule of thumb** | GPU memory usage around 60–90% is ideal |

---

### 🔄 What to Do After OOM Happens

Only the GPU stops. **System RAM is safe**.  
The whole Colab session does not crash.  
To restart:

> Menu → **Runtime** → **Restart session**

This frees the GPU memory.





#### Types of GPUs in Colab

| GPU Model      | Memory Capacity  | Plan / Conditions            | Rough Price (USD / hour)          |
| ----------- | ------ | ----------------- | --------------------- |
| NVIDIA T4   | About 15 GB | Free plan (up to 12 hours)     | Free                    |
| NVIDIA L4   | About 24 GB | Paid plan (there may be a free plan for students in the U.S.) | About **0.18 USD / hour** (rough estimate) |
| NVIDIA A100 | About 40 GB | Paid plan (there may be a free plan for students in the U.S.) | About **0.76 USD / hour** (rough estimate) |

Note: Actual billing rates and which GPU you get depend on your account, the time of day, and availability.

#### 🙋 Should You Pay?!

Conclusion: Yes, pay for it.

First, the A100 40GB is incredibly cheap for what it gives you.

Think about your tuition. Skimping here is not the smart move.

The A100 is about 3~10 times faster than the T4, so if you think your own hourly value is more than 1 USD, pay for it.

Also, for the whole curriculum, one purchase (10 USD) is enough.

Note: Credits expire after 90 days, so if you want to avoid buying again, finish within 3 months!


- About the paid plan

You **do not need** the fixed monthly Colab Pro plan.

Choose the pay-as-you-go plan, [Pay as you Go](https://colab.research.google.com/signup?utm_source=resource_tab&utm_medium=link&utm_campaign=payg_learn_more).

---

#### 💸 But I Still Don’t Want to Pay 🥺

No worries.
This curriculum is **totally fine with only the free T4 GPU!**

Even without paying, you can build a full-scratch LLM that generates old-style stories like the one shown in Chapter13 ✨

---

#### ⚠️ About the Limits of Colab’s Free Tier

Colab’s T4 GPU has **private limits**.

If you go through about 10 Chapters in 2 weeks, the GPU may become temporarily unavailable 😳

But it is okay 👇


#### ⚙️ What to Do When Colab’s Free Tier Runs Out

| Option                                                     | Details                                 | Downside               | Recommendation        |
| :----------------------------------------------------- | :--------------------------------- | :------------------ | :--------- |
| 💰 **Pay only 10 USD ([Pay as you go](https://colab.research.google.com/signup?utm_source=resource_tab&utm_medium=link&utm_campaign=payg_learn_more))**                                 | About 10 USD gives you 50+ hours of T4 GPU. More than enough for the whole curriculum. | Costs a little money 💸          | ⭐⭐⭐⭐ |
| 🆕 **Use another Google account**                                | You can reuse the free tier. Simple and reliable.                  | Drive files get split across accounts and become harder to manage.   | ⭐⭐⭐        |
| 💻 **[Kaggle Notebooks](https://www.kaggle.com/code)** | Free GPU available (phone verification required).                    | Google Drive integration is a huge pain | ⭐     |


#### 🧯 Don’t Just Close the Tab. Disconnect the Session!

**Even if you close your laptop, the GPU keeps running ⚡**

“All done! Laptop closed 💻⤵️”<br>
“All done! I’ll close Colab and watch YouTube ▶️”

↑ This is the worst move! Always disconnect the session!

Even if you close your laptop or close the Colab tab, the session keeps running!

Be careful, because it will waste your precious free quota!

When you finish, click **🔽** in the top-right corner → **Disconnect runtime**!

---

### ✅ Summary

| What to do              | Why     |
| :---------------- | :----- |
| T4 GPU is OK         | Free is enough  |
| If you hit the limit, use another account | You can keep going   |
| Disconnect when done         | Save GPU time! |

Now your **free GPU life** is perfect 💪🚀

---

### ⚠️ To the Wonderful People Paying for A100

Even if you measure `tokens_per_minute` on an A100, you will not see a big difference from the T4 GPU.<br>
The A100’s advantage over the T4 is matrix computation, but with this Config, the model size is too small,<br>
and the matrices are also too small, so the A100 cannot show its strength.<br>
Once the model size becomes large enough, it will be about 3~10 times faster than the T4.<br>


**Section 2: Out Of Memory Error** <label><input type="checkbox"> Mark as Done</label>

**`Chapter 14: Tokens per second(T4 GPU)`** <label><input type="checkbox"> Mark as Done</label>